<a href="https://colab.research.google.com/github/JMCVALE/ASN-ROCKS-/blob/main/RedesNeurais_hiperparametros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**1) Iris Dataset (3 classes de target, 150 amostras)**
Classificação — (3 classes de target, 150 amostras)

### **Exercício 1a** - Ajuste um modelo de redes neurais MLP para resolução do problema do Íris (classificação de amostras entre 3 possíveis categorias), determine a quantidade de camadas ocultas e neurônios, bem como a função de ativação da(s) camada(s) oculta(s) (sugestão: teste diferentes valores para encontrar o de melhor desempenho). Configure para utilizar early stopping e defina 500 iterações máximas.

hidden_layer_sizes : Define a arquitetura da rede neural, ou seja, quantos neurônios e camadas ocultas ela terá.

activation: Define a função de ativação usada nas camadas ocultas.

early_stopping: Controla se o treinamento deve parar automaticamente quando o desempenho em validação não melhora.

solver: Define o otimizador usado para ajustar os pesos da rede neural.

max_iter: Define o número máximo de iterações (épocas) para o treinamento.

random_state: Controla a aleatoriedade dos pesos iniciais e do embaralhamento dos dados.

In [ ]:
# Inicialização
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Carregar base
X, y = load_iris(return_X_y=True)
iris = load_iris()

# Converter para DataFrame
df = pd.DataFrame(X, columns=iris.feature_names)
df["target"] = y

# Ver primeiras linhas
print(df.head())

# Resumo
print(iris.DESCR)  # descrição detalhada da base
print(df.info())   # resumo das colunas e tipos
print(df['target'].value_counts())  # quantas amostras em cada classe

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  
.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Stat

Dividir a base em treino e teste.

In [ ]:
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Padronizar as variáveis porque estão em escalas bem diferentes

In [ ]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

Configurar os hiperparâmetros da rede neural

In [ ]:
# Configure os parâmetros corretamente

clf = MLPClassifier(hidden_layer_sizes = (64, 2),
                    activation = 'relu',
                    early_stopping = True,
                    solver = 'sgd',
                    max_iter = 500,
                    random_state=42)
clf.fit(X_tr_s, y_tr)

y_pred = clf.predict(X_te_s)
print("Acurácia:", f"{accuracy_score(y_te, y_pred):.2%}")
print(classification_report(y_te, y_pred, digits=3))

Acurácia: 60.00%
              precision    recall  f1-score   support

           0      0.833     1.000     0.909        10
           1      0.444     0.800     0.571        10
           2      0.000     0.000     0.000        10

    accuracy                          0.600        30
   macro avg      0.426     0.600     0.494        30
weighted avg      0.426     0.600     0.494        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testando diferentes configurações de hiperparâmetros e early stopping

In [ ]:
# Configurações para testar
configs = [
    {"hidden_layer_sizes": (64, 2), "activation": "relu", "solver": "sgd"},
    {"hidden_layer_sizes": (8,4), "activation": "tanh", "solver": "sgd"},
    {"hidden_layer_sizes": (2,1), "activation": "tanh", "solver": "sgd"},
    {"hidden_layer_sizes": (15,2), "activation": "logistic", "solver": "sgd"},
    {"hidden_layer_sizes": (20, 5), "activation": "relu", "solver": "sgd"},
]

# Rodar os testes
results = []
for i, cfg in enumerate(configs, 1):
    clf = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        activation=cfg["activation"],
        solver=cfg["solver"],
        early_stopping=True,
        max_iter=500,
        random_state=42
    )

    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    acc = accuracy_score(y_te, y_pred)

    results.append({
        "Teste": i,
        "Camadas Ocultas": cfg["hidden_layer_sizes"],
        "Ativação": cfg["activation"],
        "Solver": cfg["solver"],
        "Acurácia": round(acc * 100, 2)
    })

    print(f"=== Teste {i} ===")
    print(f"Configuração: {cfg}")
    print(f"Acurácia: {acc*100:.2f}%")
    print(classification_report(y_te, y_pred, digits=2))
    print("-"*50)

# 5️⃣ Mostrar resumo dos resultados
df_results = pd.DataFrame(results)
print("\nResumo dos testes:")
display(df_results)

=== Teste 1 ===
Configuração: {'hidden_layer_sizes': (64, 2), 'activation': 'relu', 'solver': 'sgd'}
Acurácia: 60.00%
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        10
           1       0.44      0.80      0.57        10
           2       0.00      0.00      0.00        10

    accuracy                           0.60        30
   macro avg       0.43      0.60      0.49        30
weighted avg       0.43      0.60      0.49        30

--------------------------------------------------
=== Teste 2 ===
Configuração: {'hidden_layer_sizes': (8, 4), 'activation': 'tanh', 'solver': 'sgd'}
Acurácia: 53.33%
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        10
           1       0.40      0.80      0.53        10
           2       0.00      0.00      0.00        10

    accuracy                           0.53        30
   macro avg       0.40      0.53      0.44        30
weighte

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

,Teste,Camadas Ocultas,Ativação,Solver,Acurácia
0,1,"(64, 2)",relu,sgd,60.00
1,2,"(8, 4)",tanh,sgd,53.33
2,3,"(2, 1)",tanh,sgd,33.33
3,4,"(15, 2)",logistic,sgd,33.33
4,5,"(20, 5)",relu,sgd,33.33


Seguindo o enunciado do Exercício 1a, foi ajustado um modelo de rede neural MLP para o problema de classificação do conjunto de dados Iris, testando diferentes combinações de camadas ocultas, neurônios e funções de ativação. Todas as configurações utilizaram o otimizador sgd (Stochastic Gradient Descent), limite de 500 iterações e o parâmetro early_stopping=True, conforme solicitado. Foram avaliadas cinco arquiteturas distintas, variando entre as funções de ativação relu, tanh e logistic.

Os resultados mostraram acurácias entre 33,33% e 60%, indicando que, com o early stopping ativado, o modelo interrompeu o treinamento antes de atingir o melhor ponto de aprendizado. A melhor configuração foi (64, 2) com função relu, atingindo 60% de acurácia. Esse comportamento demonstra que o uso do early_stopping, embora útil para evitar overfitting, pode causar underfitting em bases pequenas como a Iris, pois o treinamento é interrompido precocemente. Dessa forma, os testes evidenciam como a quantidade de camadas, a função de ativação e o controle de parada influenciam diretamente o desempenho do modelo na classificação multiclasse.

In [ ]:
ativacoes = ['identity', 'logistic', 'tanh', 'relu']

for a in ativacoes:
    clf = MLPClassifier(hidden_layer_sizes=(10,10),
                        activation=a,
                        solver='sgd',
                        early_stopping=False,
                        max_iter=800,
                        random_state=42)
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    acc = accuracy_score(y_te, y_pred)
    print(f"Ativação: {a} → Acurácia: {acc*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (800) reached and the optimization hasn't converged yet.
  warnings.warn(


Ativação: identity → Acurácia: 96.67%
Ativação: logistic → Acurácia: 33.33%


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (800) reached and the optimization hasn't converged yet.
  warnings.warn(


Ativação: tanh → Acurácia: 93.33%
Ativação: relu → Acurácia: 93.33%


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (800) reached and the optimization hasn't converged yet.
  warnings.warn(


Testando diferentes configurações de hiperparâmetros sem early stopping

In [ ]:
# Configurações para testar
configs = [
    {"hidden_layer_sizes": (64, 2), "activation": "relu", "solver": "sgd"},
    {"hidden_layer_sizes": (8,4), "activation": "tanh", "solver": "sgd"},
    {"hidden_layer_sizes": (2,1), "activation": "tanh", "solver": "sgd"},
    {"hidden_layer_sizes": (15,2), "activation": "logistic", "solver": "sgd"},
    {"hidden_layer_sizes": (20, 5), "activation": "tanh", "solver": "sgd"},
]

# Rodar os testes
results = []
for i, cfg in enumerate(configs, 1):
    clf = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        activation=cfg["activation"],
        solver=cfg["solver"],
        early_stopping=False,
        max_iter=500,
        random_state=42
    )

    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    acc = accuracy_score(y_te, y_pred)

    results.append({
        "Teste": i,
        "Camadas Ocultas": cfg["hidden_layer_sizes"],
        "Ativação": cfg["activation"],
        "Solver": cfg["solver"],
        "Acurácia": round(acc * 100, 2)
    })

    print(f"=== Teste {i} ===")
    print(f"Configuração: {cfg}")
    print(f"Acurácia: {acc*100:.2f}%")
    print(classification_report(y_te, y_pred, digits=2))
    print("-"*50)

# Mostrar resumo dos resultados
df_results = pd.DataFrame(results)
print("\nResumo dos testes:")
display(df_results)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bei

=== Teste 1 ===
Configuração: {'hidden_layer_sizes': (64, 2), 'activation': 'relu', 'solver': 'sgd'}
Acurácia: 66.67%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.50      1.00      0.67        10
           2       0.00      0.00      0.00        10

    accuracy                           0.67        30
   macro avg       0.50      0.67      0.56        30
weighted avg       0.50      0.67      0.56        30

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


=== Teste 2 ===
Configuração: {'hidden_layer_sizes': (8, 4), 'activation': 'tanh', 'solver': 'sgd'}
Acurácia: 70.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.56      0.50      0.53        10
           2       0.55      0.60      0.57        10

    accuracy                           0.70        30
   macro avg       0.70      0.70      0.70        30
weighted avg       0.70      0.70      0.70        30

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bei

=== Teste 3 ===
Configuração: {'hidden_layer_sizes': (2, 1), 'activation': 'tanh', 'solver': 'sgd'}
Acurácia: 70.00%
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.10      0.18        10
           2       0.56      1.00      0.71        10

    accuracy                           0.70        30
   macro avg       0.82      0.70      0.62        30
weighted avg       0.82      0.70      0.62        30

--------------------------------------------------
=== Teste 4 ===
Configuração: {'hidden_layer_sizes': (15, 2), 'activation': 'logistic', 'solver': 'sgd'}
Acurácia: 33.33%
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.00      0.00      0.00        10
           2       0.33      1.00      0.50        10

    accuracy                           0.33        30
   macro avg       0.11      0.33      0.17        30
wei

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,Teste,Camadas Ocultas,Ativação,Solver,Acurácia
0,1,"(64, 2)",relu,sgd,66.67
1,2,"(8, 4)",tanh,sgd,70.00
2,3,"(2, 1)",tanh,sgd,70.00
3,4,"(15, 2)",logistic,sgd,33.33
4,5,"(20, 5)",tanh,sgd,90.00


Segui o exercício ajustando um MLP para o Iris e comparei cinco arquiteturas variando camadas e ativações, mantendo solver='sgd' e max_iter=500. Ao retirar o early stopping (early_stopping=False), o modelo pôde treinar até o limite de iterações usando 100% do conjunto de treino, o que reduziu underfitting e elevou a acurácia. A melhor configuração foi hidden_layer_sizes=(20, 5), activation='tanh', alcançando 90%. As ativações tanh tiveram desempenho superior às alternativas, enquanto logistic permaneceu fraca (33,33%).

### **Exercício 1b** - De acordo com o tipo de target, qual a função de custo utilizada?

O target do problema Iris é categórico multiclasse, pois representa três classes distintas de flores (setosa, versicolor e virginica). Para esse tipo de problema de classificação, o MLPClassifier utiliza como função de custo a entropia cruzada (cross-entropy loss), também chamada de log loss.

Essa função mede a diferença entre as probabilidades previstas pelo modelo e os valores reais das classes, penalizando previsões erradas com maior intensidade. Ela é ideal para tarefas de classificação porque ajusta o modelo de forma a maximizar a probabilidade da classe correta. Em resumo, o MLP tenta minimizar a cross-entropy loss para melhorar a precisão na predição das categorias.

In [ ]:
help(MLPClassifier)

Help on class MLPClassifier in module sklearn.neural_network._multilayer_perceptron:

class MLPClassifier(sklearn.base.ClassifierMixin, BaseMultilayerPerceptron)
 |  MLPClassifier(hidden_layer_sizes=(100,), activation='relu', *, solver='adam', alpha=0.0001, batch_size='auto', learning_rate='constant', learning_rate_init=0.001, power_t=0.5, max_iter=200, shuffle=True, random_state=None, tol=0.0001, verbose=False, warm_start=False, momentum=0.9, nesterovs_momentum=True, early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, n_iter_no_change=10, max_fun=15000)
 |
 |  Multi-layer Perceptron classifier.
 |
 |  This model optimizes the log-loss function using LBFGS or stochastic
 |  gradient descent.
 |
 |  .. versionadded:: 0.18
 |
 |  Parameters
 |  ----------
 |  hidden_layer_sizes : array-like of shape(n_layers - 2,), default=(100,)
 |      The ith element represents the number of neurons in the ith
 |      hidden layer.
 |
 |  activation : {'identity', '

### **Exercício 1c** - Qual o objetivo do parâmetro early stopping?

O parâmetro early_stopping tem como objetivo interromper o treinamento da rede neural antes que ela comece a superajustar os dados de treino (overfitting). Quando ele está ativado (early_stopping=True), o modelo reserva automaticamente uma parte dos dados de treino (10% por padrão) para validação e monitora o desempenho nessa amostra.

Durante o treinamento, se o desempenho (por exemplo, a acurácia ou o erro de validação) parar de melhorar após um certo número de iterações consecutivas, o processo de aprendizado é interrompido automaticamente — mesmo que o número máximo de iterações (max_iter) ainda não tenha sido alcançado. Assim, o early stopping ajuda a evitar que o modelo aprenda ruídos e padrões específicos do conjunto de treino, garantindo melhor generalização nos dados novos

### **Exercício 1d** - Solver foi definido como "sgd", i.e. Stochastic Gradient Descent. O que é o método Gradient Descent?

O Gradient Descent tradicional calcula o gradiente da função de custo usando todo o conjunto de dados de uma vez antes de atualizar os pesos. Isso faz com que cada passo seja mais preciso, porém o processo é mais lento e pesado computacionalmente, especialmente em bases grandes. Ele busca reduzir o erro global de forma estável, mas pode ficar preso em mínimos locais e demora mais para convergir.

Já o Stochastic Gradient Descent (SGD) faz o oposto: ele atualiza os pesos a cada amostra (ou pequeno lote) de dados, em vez de esperar processar todo o conjunto. Isso torna o treinamento muito mais rápido e dinâmico, introduzindo pequenas flutuações no caminho da otimização. Essas variações fazem o modelo escapar de mínimos locais e aprender padrões mais gerais, embora o processo seja menos estável e o erro oscile mais durante o aprendizado.

#**2) Progressão de Diabetes**
Regressão — (1 target numérico, 442 amostras, 10 features)

### **Exercício 2a** - Ajuste um modelo de redes neurais MLP para resolução do problema de progressão de diabetes, determine a quantidade de camadas ocultas e neurônios, bem como a função de ativação da(s) camada(s) oculta(s) (sugestão: teste diferentes valores para encontrar o de melhor desempenho). Configure para utilizar early stopping e defina 1000 iterações máximas.

In [6]:
# Inicialização
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


# Carregar base
X, y = load_diabetes(return_X_y=True)
diabetes = load_diabetes()

# Converter para DataFrame
df = pd.DataFrame(X, columns=diabetes.feature_names)
df["target"] = y

# Ver primeiras linhas
print(df.head())

# Resumo
print(diabetes.DESCR)  # descrição detalhada da base
print(df.info())   # resumo das colunas e tipos
print(df['target'].value_counts())  # quantas amostras em cada classe


# Treino e teste
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Padronizar variáveis
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)


# Configure os parâmetros corretamente
reg = MLPRegressor(hidden_layer_sizes=(50,), activation="relu",
                   early_stopping=True, solver='adam', max_iter=1000, random_state=42)
reg.fit(X_tr_s, y_tr)

y_pred = reg.predict(X_te_s)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
print(f"RMSE: {rmse:.3f}  |  R²: {r2_score(y_te, y_pred):.3f}")

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  
.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progres

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [24]:
# Configurações para testar
configs = [
    {"hidden_layer_sizes": (12, 2), "activation": "relu", "solver": "adam"},
    {"hidden_layer_sizes": (30, 4), "activation": "relu", "solver": "adam"},
    {"hidden_layer_sizes": (25, 3), "activation": "tanh", "solver": "adam"},
    {"hidden_layer_sizes": (25, 2), "activation": "tanh", "solver": "adam"},
    {"hidden_layer_sizes": (20, 5), "activation": "relu", "solver": "adam"},
]

# Rodar os testes
results = []
for i, cfg in enumerate(configs, 1):
    model = MLPRegressor(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        activation=cfg["activation"],
        solver=cfg["solver"],
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)

    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    r2 = float(r2_score(y_te, y_pred))

    results.append({
        "Teste": i,
        "Camadas Ocultas": cfg["hidden_layer_sizes"],
        "Ativação": cfg["activation"],
        "Solver": cfg["solver"],
        "RMSE": round(rmse, 3),
        "R²": round(r2, 3)
    })

    print(f"=== Teste {i} ===")
    print(f"Configuração: {cfg}")
    print(f"RMSE: {rmse:.3f}  |  R²: {r2:.3f}")
    print("-"*50)

# Resumo
df_results = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
print("\nResumo dos testes (ordenado por RMSE):")
print(df_results)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


=== Teste 1 ===
Configuração: {'hidden_layer_sizes': (12, 2), 'activation': 'relu', 'solver': 'adam'}
RMSE: 61.282  |  R²: 0.291
--------------------------------------------------
=== Teste 2 ===
Configuração: {'hidden_layer_sizes': (30, 4), 'activation': 'relu', 'solver': 'adam'}
RMSE: 53.978  |  R²: 0.450
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


=== Teste 3 ===
Configuração: {'hidden_layer_sizes': (25, 3), 'activation': 'tanh', 'solver': 'adam'}
RMSE: 154.422  |  R²: -3.501
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


=== Teste 4 ===
Configuração: {'hidden_layer_sizes': (25, 2), 'activation': 'tanh', 'solver': 'adam'}
RMSE: 157.509  |  R²: -3.683
--------------------------------------------------
=== Teste 5 ===
Configuração: {'hidden_layer_sizes': (20, 5), 'activation': 'relu', 'solver': 'adam'}
RMSE: 55.857  |  R²: 0.411
--------------------------------------------------

Resumo dos testes (ordenado por RMSE):
   Teste Camadas Ocultas Ativação Solver     RMSE     R²
0      2         (30, 4)     relu   adam   53.978  0.450
1      5         (20, 5)     relu   adam   55.857  0.411
2      1         (12, 2)     relu   adam   61.282  0.291
3      3         (25, 3)     tanh   adam  154.422 -3.501
4      4         (25, 2)     tanh   adam  157.509 -3.683


Os resultados mostram que as redes neurais com função de ativação ReLU apresentaram desempenho consistentemente superior às configuradas com tanh. O melhor modelo foi o da configuração (30, 4) com ativação ReLU, que obteve o menor erro médio (RMSE ≈ 53,98) e o maior coeficiente de determinação (R² ≈ 0,45), indicando que explicou cerca de 45 % da variabilidade dos dados. As demais arquiteturas com ReLU também tiveram bom desempenho, enquanto as redes com tanh não convergiram adequadamente, resultando em valores de R² negativos e erros elevados (RMSE acima de 150), evidenciando underfitting e dificuldade de aprendizado. Assim, para este conjunto de dados de regressão, a função ReLU demonstrou ser a mais eficaz, combinando estabilidade e melhor capacidade preditiva.

### **Exercício 2b** - De acordo com o tipo de target, qual a função de custo utilizada?


Como o target é contínuo, o MLPRegressor utiliza a função de custo do erro quadrático médio (MSE – Mean Squared Error) para treinar o modelo.

### **Exercício 2c** - Solver foi definido como "adam", i.e. Adaptative Moment Estimation. Como o algoritmo Adam se compara com o Gradient Descent?

O algoritmo Adam (Adaptive Moment Estimation) é uma variação avançada do Gradient Descent que combina as vantagens do momentum e da adaptação da taxa de aprendizado. Enquanto o Gradient Descent tradicional utiliza uma única taxa de aprendizado fixa e se baseia apenas no gradiente atual para ajustar os pesos, o Adam calcula médias móveis dos gradientes e de seus quadrados, permitindo que cada parâmetro tenha sua própria taxa de atualização. Essa abordagem torna o processo de treinamento mais rápido, estável e menos sensível à escolha de hiperparâmetros, sendo especialmente eficaz em problemas complexos e com dados ruidosos, como os que envolvem redes neurais.



### **Exercício 2d** - Que técnicas de tuning de hiperparâmetro poderiam ser utilizadas para encontrar os melhores valores dos hiperparâmetros do modelo?

Para encontrar os melhores valores dos hiperparâmetros do modelo, podem ser aplicadas diferentes técnicas de tuning. Entre as mais utilizadas estão a Grid Search, que realiza uma busca exaustiva testando todas as combinações possíveis dos parâmetros definidos, e a Random Search, que seleciona combinações aleatórias dentro de intervalos pré-estabelecidos, reduzindo o custo computacional. Métodos mais avançados incluem a Bayesian Search, que utiliza princípios de otimização bayesiana para explorar o espaço de busca de forma inteligente, priorizando combinações promissoras, e a Hyperband Search, que combina amostragem aleatória com um mecanismo adaptativo de alocação de recursos, interrompendo precocemente as configurações menos eficazes. Essas técnicas, geralmente associadas à validação cruzada, permitem identificar de forma eficiente os hiperparâmetros que maximizam o desempenho do modelo.